# Modular SDXL Upscale — Demo Notebook

Tiled image upscaling for Stable Diffusion XL using MultiDiffusion latent-space blending.

Loads custom blocks from [akshan-main/modular-sdxl-upscale](https://huggingface.co/akshan-main/modular-sdxl-upscale) on HuggingFace Hub.

**Requirements:** Colab GPU runtime (T4 or better), ~10GB VRAM for 2x, ~14GB for 4x progressive.

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers.git transformers accelerate safetensors matplotlib

In [ ]:
import io, os, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import ModularPipelineBlocks, ControlNetModel
from diffusers.utils import load_image

torch.set_grad_enabled(False)

def show(images, titles, figsize=(20, 7)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1: axes = [axes]
    for img, ax, title in zip(images, axes, titles):
        ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis("off")
    plt.tight_layout(); plt.show()

def run(pipe, image, **kwargs):
    defaults = dict(
        prompt="high quality, detailed, sharp",
        control_image=image,
        controlnet_conditioning_scale=1.0,
        upscale_factor=2.0,
        latent_tile_size=64,
        latent_overlap=16,
        strength=0.3,
        num_inference_steps=20,
        guidance_scale=7.5,
        output="images",
    )
    defaults.update(kwargs)
    defaults["image"] = image
    return pipe(**defaults)[0]

print("Imports OK, CUDA:", torch.cuda.is_available())

In [ ]:
# Load blocks from HuggingFace Hub
blocks = ModularPipelineBlocks.from_pretrained(
    "akshan-main/modular-sdxl-upscale",
    trust_remote_code=True,
)

# Load SDXL base model
pipe = blocks.init_pipeline("stabilityai/stable-diffusion-xl-base-1.0")
pipe.load_components(torch_dtype=torch.float16)

# Add ControlNet Tile (recommended)
controlnet = ControlNetModel.from_pretrained(
    "xinsir/controlnet-tile-sdxl-1.0", torch_dtype=torch.float16
)
pipe.update_components(controlnet=controlnet)
pipe.to("cuda")

print("Pipeline ready")
print("Has controlnet:", pipe.controlnet is not None)

In [ ]:
import urllib.request

url = "https://cdn.pixabay.com/photo/2014/11/30/14/11/cat-551554_640.jpg"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
data = urllib.request.urlopen(req).read()
test_image = Image.open(io.BytesIO(data)).convert("RGB").resize((512, 512))

# Synthetic gradient for seam testing
arr = np.zeros((512, 512, 3), dtype=np.uint8)
arr[:, :, 0] = np.linspace(0, 255, 512, dtype=np.uint8)[np.newaxis, :]
arr[:, :, 1] = np.linspace(0, 255, 512, dtype=np.uint8)[:, np.newaxis]
arr[:, :, 2] = 128
gradient = Image.fromarray(arr)

show([test_image, gradient], ["Input 512x512", "Gradient 512x512"], figsize=(12, 5))

## Test 1: 2x upscale with ControlNet Tile

The standard use case. 512x512 input upscaled to 1024x1024 with MultiDiffusion latent-space blending and ControlNet Tile for structure preservation.

In [ ]:
gen = torch.Generator("cuda").manual_seed(42)
t0 = time.time()
img_2x = run(pipe, test_image, generator=gen)
print(f"Size: {img_2x.size}, Time: {time.time()-t0:.1f}s")
show([test_image, img_2x], ["Input 512x512", "2x upscale (1024x1024)"])

## Test 2: With vs Without ControlNet

Without ControlNet Tile, the model hallucinates new content instead of faithfully upscaling. ControlNet constrains each tile to match the input structure.

In [ ]:
gen = torch.Generator("cuda").manual_seed(42)
img_no_cn = run(pipe, test_image, control_image=None, controlnet_conditioning_scale=0.0, generator=gen)
show([test_image, img_no_cn, img_2x],
     ["Input", "Without ControlNet", "With ControlNet Tile"],
     figsize=(18, 6))

## Test 3: Strength comparison

`strength` controls how much the model changes the image. Lower values preserve the input more closely, higher values add more detail but risk drift.

In [ ]:
strengths = [0.1, 0.3, 0.5, 0.7]
imgs_str = []
for s in strengths:
    gen = torch.Generator("cuda").manual_seed(42)
    imgs_str.append(run(pipe, test_image, strength=s, auto_strength=False, generator=gen))
    print(f"  strength={s} done")

show([test_image] + imgs_str, ["Input"] + [f"s={s}" for s in strengths], figsize=(28, 6))

## Test 4: 4x upscale (128x128 input)

Extreme upscale from very small input. Shows the limitation — SDXL is trained on 1024x1024, so 128px inputs lack the information for faithful 4x reconstruction.

In [ ]:
small = test_image.resize((128, 128))
gen = torch.Generator("cuda").manual_seed(42)
t0 = time.time()
img_4x = run(pipe, small, upscale_factor=4.0, control_image=small, generator=gen)
print(f"Size: {img_4x.size}, Time: {time.time()-t0:.1f}s")
show([small, img_4x], ["Input 128x128", f"4x upscale ({img_4x.size[0]}x{img_4x.size[1]})"])

## Test 5: Tile size consistency

Different `latent_tile_size` values should produce consistent output thanks to MultiDiffusion cosine-weighted blending. This verifies no tile-boundary artifacts.

In [ ]:
tile_sizes = [32, 48, 64, 96]
imgs_tile = []
for ts in tile_sizes:
    gen = torch.Generator("cuda").manual_seed(42)
    imgs_tile.append(run(pipe, test_image, latent_tile_size=ts, latent_overlap=min(8, ts//2), generator=gen))
    print(f"  tile_size={ts} done")

show([test_image] + imgs_tile, ["Input"] + [f"tile={ts}" for ts in tile_sizes], figsize=(28, 6))

## Test 6: Gradient seam test

Synthetic gradient image is the worst case for tiled processing — any seam artifacts or color banding will be immediately visible.

In [ ]:
gen = torch.Generator("cuda").manual_seed(42)
img_grad = run(pipe, gradient, control_image=gradient, strength=0.15, num_inference_steps=15, generator=gen)
show([gradient, img_grad], ["Input gradient", "Output (inspect for seams)"])

## Test 7: Progressive vs single-pass 4x

For upscale factors above 2x, `progressive=True` (default) automatically splits into multiple 2x passes. This produces much better results than a single 4x jump, especially from small inputs.

In [ ]:
small_256 = test_image.resize((256, 256))

print("Progressive 4x (2x + 2x):")
gen = torch.Generator("cuda").manual_seed(42)
t0 = time.time()
img_prog = run(pipe, small_256, upscale_factor=4.0, progressive=True, control_image=small_256, generator=gen)
print(f"  Size: {img_prog.size}, Time: {time.time()-t0:.1f}s")

print("Single-pass 4x:")
gen = torch.Generator("cuda").manual_seed(42)
t0 = time.time()
img_single = run(pipe, small_256, upscale_factor=4.0, progressive=False, control_image=small_256, generator=gen)
print(f"  Size: {img_single.size}, Time: {time.time()-t0:.1f}s")

show([small_256, img_single, img_prog],
     [f"Input {small_256.size}", f"Single-pass 4x {img_single.size}", f"Progressive 4x {img_prog.size}"],
     figsize=(18, 6))

## Available Parameters

This pipeline supports the following customizations:

- **Schedulers**: `scheduler_name="Euler"`, `"DPM++ 2M"`, `"DPM++ 2M Karras"`
- **Negative prompt**: custom or default (`use_default_negative=True`)
- **Auto-strength**: automatically scales denoise strength per pass (`auto_strength=True`)
- **Progressive upscaling**: splits 4x+ into multiple 2x passes (`progressive=True`)
- **Guidance scale**: CFG strength via `guidance_scale` (default 7.5)
- **ControlNet scale**: `controlnet_conditioning_scale` (default 1.0)

These parameters have subtle visual effects when ControlNet Tile is active at scale 1.0, which is by design — the upscaler prioritizes faithfulness to the input.

## Limitations

- **SDXL is trained on 1024x1024** — tiles larger than this (`latent_tile_size > 128`) may produce artifacts. Tiles smaller than 512px (`latent_tile_size < 64`) lose context.
- **4x from very small inputs (< 256px) produces distortion** — the model doesn't have enough information to reconstruct fine details. Use progressive mode and start from at least 256px.
- **ControlNet Tile is required for faithful upscaling** — without it, the model hallucinates new content instead of enhancing existing detail.
- **Parameters like `guidance_scale`, `strength`, and `negative_prompt` have subtle effects when ControlNet is at scale 1.0** — this is correct behavior. ControlNet constrains the output to match the input structure.
- **VRAM usage scales with tile count** — 2x upscale of 512 to 1024 needs ~10GB. 4x progressive needs ~14GB peak. Uses fp16 and VAE tiling (enabled automatically).
- **Not suitable for upscaling text, line art, or pixel art** — the diffusion model will smooth sharp edges. Use nearest-neighbor or dedicated upscalers for those.

## Validation

In [ ]:
def expect_error(name, fn):
    try:
        fn()
        print(f"FAIL {name}")
    except Exception as e:
        print(f"OK   {name}: {type(e).__name__}")

expect_error("latent_tile_size=0", lambda: pipe(
    prompt="t", image=test_image, upscale_factor=2.0,
    latent_tile_size=0, latent_overlap=8, strength=0.3,
    num_inference_steps=5, output="images"))

expect_error("latent_overlap >= tile_size", lambda: pipe(
    prompt="t", image=test_image, upscale_factor=2.0,
    latent_tile_size=64, latent_overlap=64, strength=0.3,
    num_inference_steps=5, output="images"))

print("All validation passed")

## Export PDF

In [ ]:
os.makedirs("/content", exist_ok=True)

all_results = {
    "Test 1: 2x + ControlNet Tile": [test_image, img_2x],
    "Test 2: With vs Without CN": [test_image, img_no_cn, img_2x],
    "Test 3: Strength (0.1 / 0.3 / 0.5 / 0.7)": [test_image] + imgs_str,
    "Test 4: 4x from 128px": [small, img_4x],
    "Test 5: Tile sizes (32/48/64/96)": [test_image] + imgs_tile,
    "Test 6: Gradient seam": [gradient, img_grad],
    "Test 7: Progressive vs single-pass 4x": [small_256, img_single, img_prog],
}

pdf_pages = []
for title, images in all_results.items():
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 6))
    if n == 1: axes = [axes]
    fig.suptitle(title, fontsize=16)
    for img, ax in zip(images, axes):
        ax.imshow(img); ax.axis("off")
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    pdf_pages.append(Image.open(buf).convert("RGB"))

pdf_pages[0].save("/content/modular_sdxl_upscale_results.pdf",
                   save_all=True, append_images=pdf_pages[1:])
print(f"Saved PDF ({len(pdf_pages)} pages)")